# MobileNetV2 Liveness — Training Notebook

Trains MobileNetV2 (pretrained ImageNet) on CelebA-Spoof crops with full augmentation.
Exports TorchScript checkpoint compatible with the existing inference service.

In [1]:
# !pip install -q torch torchvision opencv-python pandas numpy tqdm

In [2]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as tv_models
import torchvision.transforms as T
from tqdm import tqdm

In [3]:
@dataclass
class TrainConfig:
    train_manifest: str
    val_manifest: str
    output_dir: str
    image_size: int = 112
    batch_size: int = 64
    epochs: int = 15
    lr_backbone: float = 1e-4
    lr_head: float = 1e-3
    weight_decay: float = 1e-4
    num_workers: int = 2
    seed: int = 42

In [4]:
# ----- Dataset ----------------------------------------------------------

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def make_train_transform(image_size: int) -> T.Compose:
    return T.Compose([
        T.ToPILImage(),
        T.Resize((image_size, image_size)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=10),
        T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.05),
        T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def make_val_transform(image_size: int) -> T.Compose:
    return T.Compose([
        T.ToPILImage(),
        T.Resize((image_size, image_size)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


class ManifestDataset(Dataset):
    LEGACY_ROOT = Path('/kaggle/working/celeba_spoof_prepared_full')

    def __init__(self, manifest_path: str, transform: T.Compose) -> None:
        self.df = pd.read_csv(manifest_path)
        self.transform = transform
        self.prepared_root = Path(manifest_path).parent.parent

    def __len__(self) -> int:
        return len(self.df)

    def _resolve(self, raw: str) -> Path:
        p = Path(raw)
        if p.exists():
            return p
        legacy = str(self.LEGACY_ROOT) + '/'
        if raw.startswith(legacy):
            candidate = self.prepared_root / raw[len(legacy):]
            if candidate.exists():
                return candidate
        marker = 'crops_80x80/'
        if marker in raw:
            candidate = self.prepared_root / 'crops_80x80' / raw.split(marker, 1)[1]
            if candidate.exists():
                return candidate
        return p

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_bgr = cv2.imread(str(self._resolve(row.image_path)))
        if img_bgr is None:
            raise RuntimeError(f'Cannot load: {row.image_path}')
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        tensor = self.transform(img_rgb)
        return tensor, int(row.label)

In [5]:
# ----- Model (MobileNetV2) ---------------------------------------------

def build_mobilenetv2_fas(pretrained: bool = True) -> nn.Module:
    weights = 'IMAGENET1K_V1' if pretrained else None
    backbone = tv_models.mobilenet_v2(weights=weights)
    backbone.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(1280, 2),
    )
    return backbone

In [6]:
# ----- Metrics ----------------------------------------------------------

def compute_acer(scores: list, labels: list, threshold: float = 0.5) -> float:
    """Average Classification Error Rate = (APCER + BPCER) / 2."""
    tp = fp = tn = fn = 0
    for s, l in zip(scores, labels):
        pred = 1 if s >= threshold else 0
        if l == 1 and pred == 1: tp += 1
        elif l == 0 and pred == 1: fp += 1
        elif l == 0 and pred == 0: tn += 1
        else: fn += 1
    apcer = fp / max(fp + tn, 1)
    bpcer = fn / max(fn + tp, 1)
    return (apcer + bpcer) / 2


def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict:
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss = total_correct = total_count = 0
    all_scores: list = []
    all_labels: list = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)[:, 1]
            total_loss += float(loss.item()) * labels.size(0)
            total_correct += int((logits.argmax(1) == labels).sum().item())
            total_count += int(labels.size(0))
            all_scores.extend(probs.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    return {
        'loss': total_loss / max(total_count, 1),
        'acc': total_correct / max(total_count, 1),
        'acer': compute_acer(all_scores, all_labels, threshold=0.5),
    }

In [7]:
# ----- Paths (Kaggle) ---------------------------------------------------

MANIFEST_ROOT = Path(
    '/kaggle/input/datasets/doraemongwa/celeba-spoof-prepared-full'
    '/celeba_spoof_prepared_full/manifests'
)
OUT_ROOT = Path('/kaggle/working/mobilenetv2_fas_training')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

cfg = TrainConfig(
    train_manifest=str(MANIFEST_ROOT / 'train.csv'),
    val_manifest=str(MANIFEST_ROOT / 'val.csv'),
    output_dir=str(OUT_ROOT),
)
print(cfg)
print('train manifest exists:', Path(cfg.train_manifest).exists())
print('val manifest   exists:', Path(cfg.val_manifest).exists())

TrainConfig(train_manifest='/kaggle/input/datasets/doraemongwa/celeba-spoof-prepared-full/celeba_spoof_prepared_full/manifests/train.csv', val_manifest='/kaggle/input/datasets/doraemongwa/celeba-spoof-prepared-full/celeba_spoof_prepared_full/manifests/val.csv', output_dir='/kaggle/working/mobilenetv2_fas_training', image_size=112, batch_size=64, epochs=15, lr_backbone=0.0001, lr_head=0.001, weight_decay=0.0001, num_workers=2, seed=42)
train manifest exists: True
val manifest   exists: True


In [8]:
# ----- Data loaders -----------------------------------------------------

torch.manual_seed(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

train_ds = ManifestDataset(cfg.train_manifest, make_train_transform(cfg.image_size))
val_ds   = ManifestDataset(cfg.val_manifest,   make_val_transform(cfg.image_size))

labels_list = train_ds.df['label'].tolist()
n_live  = sum(1 for l in labels_list if l == 1)
n_spoof = sum(1 for l in labels_list if l == 0)
print(f'train: {n_live} live, {n_spoof} spoof')
sample_weights = [1.0 / n_live if l == 1 else 1.0 / n_spoof for l in labels_list]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights))

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    sampler=sampler,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
)

device: cuda
train: 298920 live, 146044 spoof


In [9]:
# ----- Model + optimizer ------------------------------------------------

model = build_mobilenetv2_fas(pretrained=True).to(device)

optimizer = torch.optim.AdamW([
    {'params': model.features.parameters(),    'lr': cfg.lr_backbone},
    {'params': model.classifier.parameters(),  'lr': cfg.lr_head},
], weight_decay=cfg.weight_decay)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.epochs, eta_min=1e-6
)
criterion = nn.CrossEntropyLoss()
best_ckpt = Path(cfg.output_dir) / 'best_mobilenetv2.pt'

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 111MB/s] 


In [10]:
# ----- Training loop ----------------------------------------------------

best_acer = 1.0
patience_left = 5
history: list = []

for epoch in range(1, cfg.epochs + 1):
    model.train()
    train_loss = train_correct = train_count = 0

    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{cfg.epochs}'):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += float(loss.item()) * labels.size(0)
        train_correct += int((logits.argmax(1) == labels).sum().item())
        train_count += int(labels.size(0))

    scheduler.step()

    val_metrics = evaluate(model, val_loader, device)
    row = {
        'epoch': epoch,
        'train_loss': train_loss / max(train_count, 1),
        'train_acc':  train_correct / max(train_count, 1),
        'val_loss':   val_metrics['loss'],
        'val_acc':    val_metrics['acc'],
        'val_acer':   val_metrics['acer'],
    }
    history.append(row)
    print(row)

    if val_metrics['acer'] < best_acer:
        best_acer = val_metrics['acer']
        patience_left = 5
        torch.save({'state_dict': model.state_dict(), 'image_size': cfg.image_size}, best_ckpt)
        print(f'  -> new best ACER {best_acer:.4f}, checkpoint saved')
    else:
        patience_left -= 1
        if patience_left == 0:
            print('Early stopping.')
            break

(Path(cfg.output_dir) / 'history.json').write_text(json.dumps(history, indent=2))
print('Best val ACER:', best_acer)

Epoch 1/15: 100%|██████████| 6953/6953 [35:00<00:00,  3.31it/s]


{'epoch': 1, 'train_loss': 0.05840308759957768, 'train_acc': 0.9786274844706538, 'val_loss': 0.03365285618989472, 'val_acc': 0.9876011326860842, 'val_acer': 0.00997077412497952}
  -> new best ACER 0.0100, checkpoint saved


Epoch 2/15: 100%|██████████| 6953/6953 [30:52<00:00,  3.75it/s]


{'epoch': 2, 'train_loss': 0.030433835851009824, 'train_acc': 0.9893766686743197, 'val_loss': 0.014069253673647383, 'val_acc': 0.9950849514563107, 'val_acer': 0.005379103661675503}
  -> new best ACER 0.0054, checkpoint saved


Epoch 3/15: 100%|██████████| 6953/6953 [28:38<00:00,  4.05it/s]


...

{'epoch': 13, 'train_loss': 0.0028227643240961737, 'train_acc': 0.9990089085858631, 'val_loss': 0.004999621389239455, 'val_acc': 0.9983818770226537, 'val_acer': 0.001578824672926311}
  -> new best ACER 0.0016, checkpoint saved


Epoch 14/15: 100%|██████████| 6953/6953 [26:59<00:00,  4.29it/s]


{'epoch': 14, 'train_loss': 0.0025562100179922984, 'train_acc': 0.9991347614638487, 'val_loss': 0.0051861457058538715, 'val_acc': 0.9984021035598706, 'val_acer': 0.0016861869198058926}


Epoch 15/15: 100%|██████████| 6953/6953 [28:55<00:00,  4.01it/s]


{'epoch': 15, 'train_loss': 0.002133866653605302, 'train_acc': 0.9992471301049074, 'val_loss': 0.004730467140916206, 'val_acc': 0.998462783171521, 'val_acer': 0.001671388574130718}
Best val ACER: 0.001578824672926311


In [11]:
# ----- Export TorchScript -----------------------------------------------

payload = torch.load(best_ckpt, map_location='cpu')
model_cpu = build_mobilenetv2_fas(pretrained=False)
model_cpu.load_state_dict(payload['state_dict'])
model_cpu.eval()

scripted_path = Path(cfg.output_dir) / 'mobilenetv2_fas_scripted.pt'
scripted = torch.jit.script(model_cpu)
scripted.save(str(scripted_path))

summary = {
    'best_acer': float(best_acer),
    'image_size': cfg.image_size,
    'best_checkpoint': str(best_ckpt),
    'scripted_checkpoint': str(scripted_path),
}
(Path(cfg.output_dir) / 'run_summary.json').write_text(json.dumps(summary, indent=2))
print(summary)

{'best_acer': 0.001578824672926311, 'image_size': 112, 'best_checkpoint': '/kaggle/working/mobilenetv2_fas_training/best_mobilenetv2.pt', 'scripted_checkpoint': '/kaggle/working/mobilenetv2_fas_training/mobilenetv2_fas_scripted.pt'}
